# Chest X-ray Grad-CAM: 예측 클래스 기준 시각화 (최종 임계값 적용)

학습 완료된 **DenseNet121 + U-ignore** 모델을 사용하여 MIMIC-CXR 이미지를 분석합니다.

- 소견 순서: 심비대(Cardiomegaly), 폐부종(Edema), 흉수(Pleural Effusion)
- 출력 클래스: 0=음성(Negative), 1=양성(Positive)
- **최종 양성 판정 임계값:** `[0.45, 0.40, 0.50]`
- 판정 규칙: `positive_probability >= 해당 소견 임계값`이면 양성
- 예측 클래스 기준 Grad-CAM: **임계값으로 결정한 클래스의 logit**에 대한 기여 영역을 표시
- 출력: 원본 X-ray + 소견별 Grad-CAM 오버레이 3개

> Grad-CAM은 병변의 정확한 위치나 임상적 진단을 보증하지 않습니다. 실제 환자 영상과 출력 이미지를 공개 GitHub에 업로드하지 마세요.

## 2. 라이브러리 및 실행 환경

Grad-CAM 구현에 필요한 라이브러리를 불러오고 CPU/GPU 실행 환경을 설정한다.

- PyTorch: 모델 연산 및 기울기 계산
- torchvision: DenseNet121 모델 구성
- NumPy: 이미지 및 히트맵 배열 처리
- Matplotlib: Grad-CAM 결과 시각화

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from torch import nn
from torchvision.models import densenet121

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
FINDING_NAMES = ["Cardiomegaly", "Edema", "Pleural Effusion"]
CLASS_NAMES = ["Negative", "Positive"]
THRESHOLDS = np.array([0.45, 0.40, 0.50], dtype=np.float32)
print("장치:", device, "| 임계값:", THRESHOLDS.tolist())

## 3. DenseNet121 + U-ignore 모델 구조

실제 학습한 U-ignore softmax 모델과 같은 **소견 3개 × 음성/양성 2개** 출력 구조 `(B, 3, 2)`를 사용합니다. 기존 U-multiclass `(B, 3, 3)`와 혼동하지 않습니다.

In [ ]:
class CXRModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = densenet121(weights=None)
        self.backbone.classifier = nn.Linear(self.backbone.classifier.in_features, 6)

    def forward(self, images):
        # 학습 모델의 torchvision DenseNet forward와 일치해야 합니다.
        return self.backbone(images).reshape(-1, 3, 2)

## 4. Grad-CAM 생성 원리

마지막 특징 맵에 대한 클래스별 gradient를 계산합니다. 기본 분석 대상은 각 소견의 **최종 임계값 기준 예측 클래스**입니다. 음성 판정이라면 음성 클래스에 대한 CAM이 출력되며, 이를 양성 병변의 위치로 해석하면 안 됩니다.

In [ ]:
class GradCAM:
    def __init__(self, model):
        self.model = model
        self.target_layer = model.backbone.features.norm5

        self.activations = None
        self.gradients = None

        self.forward_handle = self.target_layer.register_forward_hook(
            self.save_activations
        )

    def save_activations(self, module, inputs, output):
        # DenseNet의 이후 연산에서 원본 출력이 변경될 수 있으므로
        # clone한 텐서에 hook을 등록
        captured = output.clone()
        self.activations = captured.detach()
        captured.register_hook(self.save_gradients)
        return captured

    def save_gradients(self, grad):
        self.gradients = grad.detach()

    def generate(self, image, finding_idx, class_idx=1):
        self.model.eval()

        device = next(self.model.parameters()).device
        image = image.unsqueeze(0).to(device)

        self.model.zero_grad(set_to_none=True)
        self.activations = None
        self.gradients = None

        with torch.enable_grad():
            logits = self.model(image)

            score = logits[0, finding_idx, class_idx]
            score.backward()

        activations = self.activations
        gradients = self.gradients

        weights = gradients.mean(dim=(2, 3), keepdim=True)
        cam = (weights * activations).sum(dim=1, keepdim=True)

        cam = F.relu(cam)
        cam = F.interpolate(
            cam,
            size=image.shape[-2:],
            mode="bilinear",
            align_corners=False
        )

        cam = cam[0, 0].detach().cpu().numpy()

        cam = cam - cam.min()
        cam = cam / (cam.max() + 1e-8)

        return cam

    def close(self):
        self.forward_handle.remove()

## 5. 결과 시각화

원본 X-ray와 세 소견의 **예측 클래스 기준 Grad-CAM**을 표시합니다. 각 제목에 양성 확률, 임계값, 최종 판정 클래스를 함께 표시합니다.

In [ ]:
def show_gradcam(image, cams, positive_probs, predictions, mean, std):
    mean = np.asarray(mean, dtype=np.float32)
    std = np.asarray(std, dtype=np.float32)
    img = image.detach().cpu().permute(1, 2, 0).numpy()
    img = np.clip(img * std + mean, 0, 1)

    fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))
    axes[0].imshow(img)
    axes[0].set_title("Original X-ray")
    for i, finding in enumerate(FINDING_NAMES):
        axes[i + 1].imshow(img)
        axes[i + 1].imshow(cams[i], cmap="jet", alpha=0.4, vmin=0, vmax=1)
        axes[i + 1].set_title(
            f"{finding} - {CLASS_NAMES[int(predictions[i])]}\n"
            f"P(+)={positive_probs[i]:.3f} | Threshold={THRESHOLDS[i]:.2f}"
        )
    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

## 6. 실제 학습 모델 및 MIMIC-CXR 이미지 불러오기

GPU 서버의 학습 완료 체크포인트와 실제 영상을 사용합니다. **임시 모델이나 pilot 데이터는 사용하지 않습니다.**

In [ ]:
PROJECT_DIR = Path("/home/team30103/chest-xray-multilabel")
MODEL_PATH = (PROJECT_DIR / "runs" / "uignore_weighted_full" /
              "train_20261019_165121_403875" / "best_model.pth")
IMAGE_ROOT = Path("/home/team30103/physionet_data")
TEST_CSV = PROJECT_DIR / "data" / "processed" / "test.csv"

for label, path in [("모델", MODEL_PATH), ("Test CSV", TEST_CSV)]:
    if not path.is_file():
        raise FileNotFoundError(f"{label} 파일이 없습니다: {path}")
if not IMAGE_ROOT.is_dir():
    raise FileNotFoundError(f"이미지 폴더가 없습니다: {IMAGE_ROOT}")
print("체크포인트:", MODEL_PATH)
print("CSV:", TEST_CSV)

In [ ]:
checkpoint = torch.load(MODEL_PATH, map_location="cpu", weights_only=True)
if checkpoint.get("smoke_test", False):
    raise ValueError("테스트용 임시 체크포인트입니다.")
if "target_labels" in checkpoint and list(checkpoint["target_labels"]) != FINDING_NAMES:
    raise ValueError("체크포인트 소견 순서가 다릅니다.")
if "class_names" in checkpoint and list(checkpoint["class_names"]) != ["negative", "positive"]:
    raise ValueError("이 노트북은 U-ignore softmax 2클래스 전용입니다.")

IMAGE_SIZE = (checkpoint.get("input") or {}).get("height") or (checkpoint.get("config") or {}).get("image_size")
if not IMAGE_SIZE:
    raise ValueError("체크포인트에서 이미지 크기를 찾지 못했습니다.")
input_spec = checkpoint.get("input") or {}
mean = np.asarray(input_spec.get("mean", [0.485, 0.456, 0.406]), dtype=np.float32)
std = np.asarray(input_spec.get("std", [0.229, 0.224, 0.225]), dtype=np.float32)
if mean.shape != (3,) or std.shape != (3,) or (std <= 0).any():
    raise ValueError("정규화 통계값이 올바르지 않습니다.")

model = CXRModel()
model.load_state_dict(checkpoint["model_state_dict"], strict=True)
model.to(device).eval()

sys.path.insert(0, str(PROJECT_DIR))
import cxr_preprocessing as preprocessing
preprocessing.MEAN = tuple(float(x) for x in mean)
preprocessing.STD = tuple(float(x) for x in std)
if list(preprocessing.TARGET_LABELS) != FINDING_NAMES:
    raise ValueError("전처리 소견 순서가 다릅니다.")
print("모델 로드 완료 | epoch:", checkpoint.get("epoch"), "| 크기:", IMAGE_SIZE)

## 7. Grad-CAM 분석 이미지 선택

MIMIC-CXR Test CSV에서 이미지를 선택합니다. 저장 구조는 기존 평가 코드와 동일하게 `IMAGE_ROOT / f"{dicom_id}.jpg"` 형식을 사용합니다. 이 노트북은 **정성적 시각화용**이며 임계값을 다시 조정하는 데 사용하지 않습니다.

In [ ]:
test_df = pd.read_csv(TEST_CSV, dtype={"dicom_id": "string"})
if not {"dicom_id", *FINDING_NAMES}.issubset(test_df.columns):
    raise ValueError("Test CSV에 dicom_id 또는 소견 라벨이 없습니다.")
IMAGE_INDEX = 0  # 확인할 영상의 0-based 행 인덱스
if not 0 <= IMAGE_INDEX < len(test_df):
    raise IndexError(f"IMAGE_INDEX는 0~{len(test_df)-1} 범위여야 합니다.")
row = test_df.iloc[IMAGE_INDEX]
image_path = IMAGE_ROOT / f"{row['dicom_id']}.jpg"
if not image_path.is_file():
    raise FileNotFoundError(f"이미지를 찾지 못했습니다: {image_path}")
with Image.open(image_path) as original:
    image = preprocessing.preprocess_image(original, image_size=int(IMAGE_SIZE))
print("영상 경로:", image_path)
print("입력 크기:", tuple(image.shape))
print("정답 라벨 (불확실 포함):", row[FINDING_NAMES].tolist())

In [ ]:
# 원래 torchvision forward와 모델 forward의 수치 일치 점검
model.eval()
x = image.unsqueeze(0).to(device)
with torch.no_grad():
    original_logits = model.backbone(x).reshape(-1, 3, 2)
    updated_logits = model(x)
max_diff = (original_logits - updated_logits).abs().max().item()
print("최대 출력 차이:", max_diff)
print("출력 일치:", torch.allclose(original_logits, updated_logits, atol=1e-6, rtol=1e-5))
assert torch.allclose(original_logits, updated_logits, atol=1e-6, rtol=1e-5)

In [ ]:
model.eval()
with torch.no_grad():
    logits = model(image.unsqueeze(0).to(device))
    positive_probs = torch.softmax(logits, dim=-1)[0, :, 1].cpu().numpy()

# argmax가 아니라 확정된 소견별 임계값으로 음성/양성을 결정합니다.
predictions = (positive_probs >= THRESHOLDS).astype(np.int64)
for i, finding in enumerate(FINDING_NAMES):
    print(f"{finding}: P(+)={positive_probs[i]:.4f}, "
          f"threshold={THRESHOLDS[i]:.2f}, 판정={CLASS_NAMES[predictions[i]]}")

In [ ]:
gradcam = GradCAM(model)
try:
    cams = [
        gradcam.generate(image, finding_idx=i, class_idx=int(predictions[i]))
        for i in range(len(FINDING_NAMES))
    ]
    show_gradcam(image, cams, positive_probs, predictions, mean, std)
finally:
    gradcam.close()